In [9]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [8]:
from crewai import Agent, Crew, Task

In [2]:
import os
from utils.api_utils import get_openai_api_key,get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

In [6]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

c:\Users\91888\anaconda3\envs\crewai-env\lib\site-packages\pydantic\_internal\_generate_schema.py:623: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [10]:
# Agent 1: Venue Coordinator
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue "
    "based on event requirements",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    )
)

In [11]:
 # Agent 2: Logistics Manager
logistics_manager = Agent(
    role='Logistics Manager',
    goal=(
        "Manage all logistics for the event "
        "including catering and equipmen"
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    )
)

In [12]:
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and "
         "communicate with participants",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    )
)

In [13]:
from pydantic import BaseModel
# Define a Pydantic model for venue details 
# (demonstrating Output as Pydantic)
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

By using output_json, you can specify the structure of the output you want.
By using output_file, you can get your output in a file.
By setting human_input=True, the task will ask for human feedback (whether you like the results or not) before finalising it.

In [14]:
venue_task = Task(
    description="Find a venue in {event_city} "
                "that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
                    "venue you found to accommodate the event.",
    human_input=True,
    output_json=VenueDetails,
    output_file="venue_details.json",  
      # Outputs the venue details as a JSON file
    agent=venue_coordinator
)

In [15]:
logistics_task = Task(
    description="Coordinate catering and "
                 "equipment for an event "
                 "with {expected_participants} participants "
                 "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
                    "including catering and equipment setup.",
    human_input=True,
    async_execution=True,
    agent=logistics_manager
)

In [21]:
marketing_task = Task(
    description="Promote the {event_topic} "
                "aiming to engage at least"
                "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
                    "and attendee engagement formatted as markdown.",
    async_execution=False,
    output_file="marketing_report.md",  # Outputs the report as a text file
    agent=marketing_communications_agent
)

In [22]:
# Define the crew with agents and tasks
event_management_crew = Crew(
    agents=[venue_coordinator, 
            logistics_manager, 
            marketing_communications_agent],
    
    tasks=[venue_task, 
           logistics_task, 
           marketing_task],
    
    verbose=True
)

In [23]:
event_details = {
    'event_topic': "Tech Innovation Conference",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "San Francisco",
    'tentative_date': "2024-09-15",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Conference Hall"
}

In [24]:
result = event_management_crew.kickoff(inputs=event_details)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9015f741-cbe7-4382-8aff-d290da066451                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 250a4114-3468-4d12-9e71-31fdd5ac572f
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 250a4114-3468-4d12-9e71-31fdd5ac572f
       Status: Executing Task...
    └── 🤖 Agent: Venue Coordinator
            Status: In Progress

# Agent: Venue Coordinator
## Task: Find a venue in San Francisco that meets criteria for Tech Innovation Conference.


🚀 Crew: crew
└── 📋 Task: 250a4114-3468-4d12-9e71-31fdd5ac572f
       Status: Executing Task...
    └── 🤖 Agent: Venue Coordinator
            Status: In Progress
        └── 🧠 Thinking...

...



# Agent: Marketing and Communications Agent
## Final Answer: 
## Marketing Activities and Attendee Engagement Report
### Proven Conference Marketing Ideas:
1. **Identify Target Audience:** Segment and tailor marketing messages for different audience groups.
2. **Utilize Demographic Data:** Collect data to personalize outreach efforts and choose effective marketing channels.
3. **Build a Memorable Brand:** Consistent branding builds recognition and trust, enhance event image.
4. **Develop Unique Value Proposition:** Communicate what sets your event apart clearly.
5. **Diversify Marketing Content:** Offer various content formats to reach a wider audience.
6. **Create Shareable Content:** Design content for easy sharing to extend event reach.
7. **Develop a Conference Website:** Use high-quality images, clear information, and optimize for mobile and search engines.
8. **Implement Event Gamification:** Engage attendees through gamification elements for interactive experiences.
9. **Surve

🚀 Crew: crew
├── 📋 Task: 250a4114-3468-4d12-9e71-31fdd5ac572f
│      Assigned to: Venue Coordinator
│      Status: ✅ Completed
│   └── 🤖 Agent: Venue Coordinator
│           Status: ✅ Completed
│       ├── 🔧 Used Search the internet with Serper (1)
│       └── 🔧 Used Read website content (1)
├── 📋 Task: 1f21caf5-308f-4ff5-b42b-5381f8b38b67
│      Assigned to: Logistics Manager
│      Status: ✅ Completed
│   └── 🤖 Agent: Logistics Manager
│           Status: ✅ Completed
│       ├── 🔧 Used Search the internet with Serper (3)
│       └── 🔧 Used Read website content (3)
└── 📋 Task: 1366f467-a960-43fa-9752-ae6fd8d82228
       Status: Executing Task...
    └── 🤖 Agent: Marketing and Communications Agent
            Status: ✅ Completed
        ├── 🔧 Used Search the internet with Serper (4)
        └── 🔧 Used Read website content (4)

🚀 Crew: crew
├── 📋 Task: 250a4114-3468-4d12-9e71-31fdd5ac572f
│      Assigned to: Venue Coordinator
│      Status: ✅ Completed
│   └── 🤖 Agent: Venue Coordinator
│           Status: ✅ Completed
│       ├── 🔧 Used Search the internet with Serper (1)
│       └── 🔧 Used Read website content (1)
├── 📋 Task: 1f21caf5-308f-4ff5-b42b-5381f8b38b67
│      Assigned to: Logistics Manager
│      Status: ✅ Completed
│   └── 🤖 Agent: Logistics Manager
│           Status: ✅ Completed
│       ├── 🔧 Used Search the internet with Serper (3)
│       └── 🔧 Used Read website content (3)
└── 📋 Task: 1366f467-a960-43fa-9752-ae6fd8d82228
       Assigned to: Marketing and Communications Agent
       Status: ✅ Completed
    └── 🤖 Agent: Marketing and Communications Agent
            Status: ✅ Completed
        ├── 🔧 Used Search the internet with Serper (4)
        └── 🔧 Used Read website content (4)

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1366f467-a960-43fa-9752-ae6fd8d82228                                                                     │
│  Agent: Marketing and Communications Agent                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9015f741-cbe7-4382-8aff-d290da066451                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [25]:
import json
from pprint import pprint

with open('venue_details.json') as f:
   data = json.load(f)

pprint(data)

{'address': '255 S Airport Blvd, South San Francisco, CA 94080',
 'booking_status': 'Open for AI Industry Conference bookings',
 'capacity': 20500,
 'name': 'South San Francisco Conference Center'}


In [26]:
from IPython.display import Markdown
Markdown("marketing_report.md")

## Marketing Activities and Attendee Engagement Report
### Proven Conference Marketing Ideas:
1. **Identify Target Audience:** Segment and tailor marketing messages for different audience groups.
2. **Utilize Demographic Data:** Collect data to personalize outreach efforts and choose effective marketing channels.
3. **Build a Memorable Brand:** Consistent branding builds recognition and trust, enhance event image.
4. **Develop Unique Value Proposition:** Communicate what sets your event apart clearly.
5. **Diversify Marketing Content:** Offer various content formats to reach a wider audience.
6. **Create Shareable Content:** Design content for easy sharing to extend event reach.
7. **Develop a Conference Website:** Use high-quality images, clear information, and optimize for mobile and search engines.
8. **Implement Event Gamification:** Engage attendees through gamification elements for interactive experiences.
9. **Survey Audience Post-Event:** Gather feedback for continuous improvement and audience insights.
10. **Partner with Industry Professionals:** Collaborate with influencers and industry experts to amplify event reach.
11. **Provide an Interactive Experience:** Host interactive sessions, Q&A sessions, and engage on social media to connect with attendees.
12. **Offer Early Bird Discounts:** Incentivize early registrations with discounts and limited-time offers.

### Attendee Engagement Strategies:
- **Personalized Interactions:** Automate personalized email flows to engage attendees.
- **Interactive Experiences:** Incorporate gamification, interactive sessions, and workshops for engagement.
- **Social Media Engagement:** Use diverse content, hashtags, and run polls for audience interaction.
- **Post-Event Follow-Up:** Send personalized thank-you emails, share resources, and encourage feedback for lasting relationships.

By implementing these marketing strategies and attendee engagement tactics, the Tech Innovation Conference can attract and engage potential attendees effectively, aiming to reach the goal of engaging at least 500 participants.